In [9]:
import sys
import os
import torch
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# FIX PATH
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

print(f"✅ Root path added: {root_path}")

✅ Root path added: c:\Users\Administrator\Desktop\smart_health_monitor


In [19]:
from src import config

import src.model as model
import src.engine as engine
import src.preprocessing as preprocessing
import src.data_loader as data_loader

importlib.reload(model)
importlib.reload(engine)
importlib.reload(preprocessing)
importlib.reload(data_loader)

print("✅ Modules loaded successfully")

✅ Modules loaded successfully


In [ ]:
from src.preprocessing import full_preprocessing_and_save
import pandas as pd

try:
    X_train, X_test, y_train, y_test = full_preprocessing_and_save()

    print("--- Data Dimensions ---")
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")

    features_df = pd.DataFrame(X_train[:5])

    print("\n--- First 5 Processed Records (Scaled) ---")
    display(features_df.head())

except Exception as e:
    print(f"❌ Error during preprocessing: {e}")

✅ Advanced processing complete! 13 features ready.
📁 Processed files saved in: ../data/processed/
--- Data Dimensions ---
X_train shape: (4000, 13)
y_train shape: (4000,)

--- First 5 Processed Records (Scaled) ---


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.350166,1.00451,0.809252,1.226203,-1.017147,-0.993521,-1.087630,0.680375,0.142360,-1.040312,-1.004008,-0.991536,0.734449
1,-0.095976,-0.99551,0.910658,1.226203,0.983142,-0.993521,-0.096884,1.514935,-1.023043,0.961250,0.996008,-0.991536,0.328448
2,-1.099797,1.00451,1.458251,-0.000307,0.983142,-0.993521,-1.697320,-1.452390,0.593322,0.961250,0.996008,-0.991536,-0.552948
3,0.015559,-0.99551,-0.874091,1.226203,0.983142,1.006521,0.436595,0.680375,-1.023043,0.961250,-1.004008,-0.991536,-0.399501
4,-0.542119,-0.99551,1.275720,-0.000307,0.983142,1.006521,-0.668468,-1.174203,-1.687516,-1.040312,0.996008,1.008536,0.005914


In [22]:
from src.data_loader import get_pytorch_loaders

train_loader, test_loader = get_pytorch_loaders(
    X_train, X_test, y_train, y_test
)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Test batches: {len(test_loader)}")

✅ Train batches: 125
✅ Test batches: 32


In [23]:
# Now this will NOT throw a NameError
data_iter = iter(train_loader)
batch_X, batch_y = next(data_iter)

print("--- DataLoader Inspection ---")
print(f"Batch X Shape: {batch_X.shape}") # Should be [32, 1, M]
print(f"Batch y Shape: {batch_y.shape}") # Should be [32, 1]
print(f"First label in batch: {batch_y[0].item()}")

--- DataLoader Inspection ---
Batch X Shape: torch.Size([32, 1, 13])
Batch y Shape: torch.Size([32, 1])
First label in batch: 1.0


In [26]:
import torch
import torch.nn as nn
import torch.optim as optim

from src.model import SmartHealthLSTM  # IMPORTANT: always import from module

# =========================================================
# 1. DEVICE SETUP
# =========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =========================================================
# 2. MODEL INITIALIZATION
# =========================================================

input_dim = X_train.shape[1]

model = SmartHealthLSTM(input_size=input_dim).to(device)

# =========================================================
# 3. LOSS FUNCTION
# =========================================================

criterion = nn.BCELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print(f"✅ Model initialized successfully on: {device}")
print(f"📊 Input dimension: {input_dim}")

✅ Model initialized successfully on: cpu
📊 Input dimension: 13


In [27]:
# 4. Start the training process
print("🚀 Training in progress... Please wait.")

# We ensure 'device' is defined (lowercase)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

history = train_engine(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,  # Changed from DEVICE to device
    patience=7,    
    epochs=100     
)

print("✨ Training Complete!")

🚀 Training in progress... Please wait.
Epoch 5: Train Loss 0.0331 | Val Loss 0.0329
Epoch 10: Train Loss 0.0233 | Val Loss 0.0235
Epoch 15: Train Loss 0.0234 | Val Loss 0.0228
Epoch 20: Train Loss 0.0204 | Val Loss 0.0195
Epoch 25: Train Loss 0.0171 | Val Loss 0.0339
--- Early stopping triggered at epoch 27 ---
✨ Training Complete!


In [33]:
import os

# Create the directory if it doesn't exist
model_dir = '.././models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Define the file path
model_path = os.path.join(model_dir, 'smart_health_lstm_model.pth')

# Save the model weights
torch.save(model.state_dict(), model_path)

print(f"✅ Model successfully saved to: {model_path}")

✅ Model successfully saved to: .././models\smart_health_lstm_model.pth


In [36]:
# 1. Re-initialize the model structure
loaded_model = SmartHealthLSTM(input_size=X_train.shape[1]).to(device)

# 2. Load the saved weights
loaded_model.load_state_dict(torch.load('../models/smart_health_lstm_model.pth'))

# 3. Set to evaluation mode
loaded_model.eval()

print("🚀 Model weights loaded and ready for prediction!")

🚀 Model weights loaded and ready for prediction!
